# Massive-STEPS Bandung - Behavior Preparation

Notebook ini untuk filtering dan mapping data Massive-STEPS Bandung.
Tahap ini belum training model. Fokusnya menyiapkan data behavior agar cocok dengan arah MuterBandung.

## Tahap A - Cek File Raw

Tujuan: pastikan file tabular train, validation, dan test tersedia.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
WORKSPACE = ROOT / "MuterBandung_Behavior_Model_Workspace"
RAW_DIR = WORKSPACE / "01_Raw_Data" / "massive_steps_bandung"

tabular_files = {
    "train": RAW_DIR / "tabular_train-00000-of-00001.parquet",
    "validation": RAW_DIR / "tabular_validation-00000-of-00001.parquet",
    "test": RAW_DIR / "tabular_test-00000-of-00001.parquet",
}

rows = []
for split, path in tabular_files.items():
    df_head = pd.read_parquet(path)
    rows.append({
        "split": split,
        "exists": path.exists(),
        "rows": len(df_head),
        "columns": len(df_head.columns),
        "column_names": ", ".join(df_head.columns[:8]),
    })

pd.DataFrame(rows)

,split,exists,rows,columns,column_names
0,train,True,113058,15,"trail_id, user_id, venue_id, latitude, longitu..."
1,validation,True,16018,15,"trail_id, user_id, venue_id, latitude, longitu..."
2,test,True,32208,15,"trail_id, user_id, venue_id, latitude, longitu..."


Keputusan: file tabular dipakai sebagai basis karena punya kategori, koordinat, waktu, user, dan trail.

## Tahap B - Audit Kategori Awal

Tujuan: lihat apakah data bisa langsung dipakai atau perlu disaring dulu.

In [2]:
frames = []
for split, path in tabular_files.items():
    frame = pd.read_parquet(path)
    frame["split"] = split
    frames.append(frame)

raw = pd.concat(frames, ignore_index=True)
raw["timestamp"] = pd.to_datetime(raw["timestamp"], errors="coerce")

audit_awal = {
    "total_rows": len(raw),
    "unique_users": raw["user_id"].nunique(),
    "unique_trails": raw["trail_id"].nunique(),
    "unique_venues": raw["venue_id"].nunique(),
    "unique_categories": raw["venue_category"].nunique(),
    "missing_name_pct": round(raw["name"].isna().mean() * 100, 2),
    "missing_latitude_pct": round(raw["latitude"].isna().mean() * 100, 2),
    "min_timestamp": str(raw["timestamp"].min()),
    "max_timestamp": str(raw["timestamp"].max()),
}
print(json.dumps(audit_awal, indent=2, ensure_ascii=False))

raw["venue_category"].value_counts().head(20).rename_axis("venue_category").reset_index(name="checkin_count")

{
  "total_rows": 161284,
  "unique_users": 3377,
  "unique_trails": 55333,
  "unique_venues": 26559,
  "unique_categories": 421,
  "missing_name_pct": 23.52,
  "missing_latitude_pct": 23.52,
  "min_timestamp": "2012-04-03 18:01:00",
  "max_timestamp": "2018-10-19 09:06:00"
}


,venue_category,checkin_count
0,Home (private),18844
1,Shopping Mall,17021
2,High School,5647
3,Road,5342
4,Café,4390
5,Indonesian Restaurant,3925
6,Multiplex,3444
7,College Classroom,3102
8,Food Truck,2300
9,Office,2294


Keputusan: data tidak boleh dipakai mentah. Kategori harian seperti rumah, sekolah, kantor, dan jalan terlalu dominan.

## Tahap C - Jalankan Filtering dan Mapping

Tujuan: pisahkan kategori yang cocok untuk MuterBandung, noise, dan kategori yang masih perlu dicek.

In [3]:
import subprocess
import sys

script_path = ROOT / "Scripts" / "prepare_massive_steps_filter_mapping.py"
result = subprocess.run(
    [sys.executable, str(script_path)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
summary = json.loads(result.stdout)
print(json.dumps({
    "raw_rows": summary["raw_rows"],
    "mapping_status_counts_rows": summary["mapping_status_counts_rows"],
    "filtered_rows": summary["filtered_rows"],
    "filtered_row_pct": summary["filtered_row_pct"],
    "sequence_rows": summary["sequence_rows"],
}, indent=2, ensure_ascii=False))

{
  "raw_rows": 161284,
  "mapping_status_counts_rows": {
    "mapped_keep": 79696,
    "excluded_noise": 63910,
    "needs_review": 17678
  },
  "filtered_rows": 79696,
  "filtered_row_pct": 49.41,
  "sequence_rows": 23473
}


Keputusan: hanya `mapped_keep` yang masuk kandidat training. Noise dan needs_review ditahan dulu.

## Tahap D - Audit Hasil Mapping

Tujuan: cek komposisi hasil mapping setelah aturan diterapkan.

In [4]:
CURATED_DIR = WORKSPACE / "03_Curated"
EVAL_DIR = WORKSPACE / "04_Evaluation"
DATE_TAG = "2026-06-10"

mapping_path = CURATED_DIR / f"MASSIVE_STEPS_BANDUNG_CATEGORY_MAPPING_CANDIDATE_{DATE_TAG}.csv"
summary_path = EVAL_DIR / f"MASSIVE_STEPS_BANDUNG_FILTERING_MAPPING_SUMMARY_{DATE_TAG}.json"
mapping = pd.read_csv(mapping_path)
summary = json.loads(summary_path.read_text(encoding="utf-8"))

status_table = mapping["mapping_status"].value_counts().rename_axis("mapping_status").reset_index(name="category_count")
label_table = (
    mapping[mapping["mapping_status"].eq("mapped_keep")]
    .sort_values("checkin_count", ascending=False)
    .groupby("muterbandung_label", as_index=False)
    .agg(category_count=("venue_category", "count"), checkin_count=("checkin_count", "sum"))
    .sort_values("checkin_count", ascending=False)
)

display(status_table)
display(label_table)

,mapping_status,category_count
0,needs_review,170
1,mapped_keep,162
2,excluded_noise,89


,muterbandung_label,category_count,checkin_count
4,kuliner,85,40120
1,belanja,11,23988
3,hiburan,24,7318
6,religi,5,3130
5,penginapan,6,2077
0,alam_santai,19,2021
2,budaya_edukasi,12,1042


Keputusan: komposisi diterima untuk baseline behavior. Kuliner dan belanja wajar dominan karena ini data check-in, bukan dataset wisata kurasi.

## Tahap E - Cek Kategori Rawan Salah

Tujuan: pastikan kategori harian tidak ikut masuk kandidat karena kata pendek seperti art/bar.

In [5]:
suspicious = mapping[
    mapping["venue_category"].str.contains(
        "barber|martial|college|school|office|home|road|apartment|arts|bar|building",
        case=False,
        na=False,
    )
].sort_values("checkin_count", ascending=False)

suspicious[["venue_category", "checkin_count", "muterbandung_label", "mapping_status", "rule_reason"]].head(50)

,venue_category,checkin_count,muterbandung_label,mapping_status,rule_reason
0,Home (private),18844,NaN,excluded_noise,matched exclude rule: private_residential
2,High School,5647,NaN,excluded_noise,matched exclude rule: education_daily
3,Road,5342,NaN,excluded_noise,matched exclude rule: infrastructure_transport
7,College Classroom,3102,NaN,excluded_noise,matched exclude rule: education_daily
9,Office,2294,NaN,excluded_noise,matched exclude rule: work_office
10,Middle School,2230,NaN,excluded_noise,matched exclude rule: education_daily
13,General College & University,2120,NaN,excluded_noise,matched exclude rule: education_daily
32,College Academic Building,1330,NaN,excluded_noise,matched exclude rule: education_daily
33,Building,1326,NaN,excluded_noise,matched exclude rule: work_office
38,Salon / Barbershop,1034,NaN,excluded_noise,matched exclude rule: personal_service


Keputusan: kategori harian sudah tertahan. `Barbershop`, `apartment`, dan `college` tidak ikut kandidat training.

## Tahap F - Queue Manual Review

Tujuan: tampilkan kategori yang belum aman diputuskan otomatis.

In [6]:
needs_review_path = EVAL_DIR / f"MASSIVE_STEPS_BANDUNG_CATEGORY_MAPPING_NEEDS_REVIEW_{DATE_TAG}.csv"
needs_review = pd.read_csv(needs_review_path)
needs_review[["venue_category", "checkin_count", "unique_venues", "missing_name_pct", "missing_latitude_pct"]].head(40)

,venue_category,checkin_count,unique_venues,missing_name_pct,missing_latitude_pct
0,Grocery Store,1954,166,6.86,6.86
1,Convenience Store,755,144,8.34,8.34
2,Photography Lab,720,33,2.08,2.08
3,Event Space,599,74,7.85,7.85
4,Electronics Store,593,125,16.69,16.69
5,Pool,570,74,28.25,28.25
6,Miscellaneous Shop,524,149,15.08,15.08
7,Video Store,501,45,7.58,7.58
8,Medical Center,460,160,21.52,21.52
9,Field,413,64,9.20,9.20


Keputusan: needs_review jangan dipaksa. Jika nanti coverage kurang, kategori ini dibuka bertahap.

## Tahap G - Candidate Sequence

Tujuan: cek data sequence yang nanti bisa dipakai untuk baseline next-category.

In [7]:
sequence_path = CURATED_DIR / f"MASSIVE_STEPS_BANDUNG_CATEGORY_SEQUENCE_CANDIDATE_{DATE_TAG}.csv"
sequences = pd.read_csv(sequence_path)

sequence_audit = {
    "sequence_rows": len(sequences),
    "unique_users": sequences["user_id"].nunique(),
    "unique_trails": sequences["trail_id"].nunique(),
    "avg_sequence_length": round(sequences["sequence_length"].mean(), 2),
    "split_counts": sequences["split"].value_counts().to_dict(),
}
print(json.dumps(sequence_audit, indent=2, ensure_ascii=False))

sequences[["trail_id", "user_id", "split", "sequence_length", "history_categories", "target_next_category"]].head(15)

{
  "sequence_rows": 23473,
  "unique_users": 3244,
  "unique_trails": 23473,
  "avg_sequence_length": 2.63,
  "split_counts": {
    "train": 16556,
    "test": 4626,
    "validation": 2291
  }
}


,trail_id,user_id,split,sequence_length,history_categories,target_next_category
0,2013_1000181,41767,train,4,belanja > hiburan > belanja,kuliner
1,2013_1000186,41767,train,3,belanja > kuliner,kuliner
2,2013_1000187,41767,test,2,kuliner,belanja
3,2013_1000188,41767,train,2,belanja,kuliner
4,2013_1000190,41767,validation,2,kuliner,belanja
5,2013_1000295,41775,train,2,belanja,kuliner
6,2013_1000296,41775,train,2,kuliner,kuliner
7,2013_1000297,41775,train,2,belanja,kuliner
8,2013_1000303,41775,test,4,belanja > kuliner > kuliner,kuliner
9,2013_1000304,41775,train,2,kuliner,kuliner


Keputusan: sequence sudah cukup untuk baseline next-category. Belum dipakai untuk next-POI langsung.

## Tahap H - Output Tahap Ini

Tujuan: catat file hasil filtering dan mapping.

In [8]:
output_files = [
    mapping_path,
    CURATED_DIR / f"MASSIVE_STEPS_BANDUNG_FILTERED_CHECKINS_CANDIDATE_{DATE_TAG}.csv",
    sequence_path,
    needs_review_path,
    EVAL_DIR / f"MASSIVE_STEPS_BANDUNG_EXCLUDED_NOISE_CATEGORIES_{DATE_TAG}.csv",
    summary_path,
]

pd.DataFrame([
    {"file": str(path.relative_to(ROOT)), "exists": path.exists(), "size_mb": round(path.stat().st_size / 1024 / 1024, 2)}
    for path in output_files
])

,file,exists,size_mb
0,MuterBandung_Behavior_Model_Workspace\03_Curat...,True,0.04
1,MuterBandung_Behavior_Model_Workspace\03_Curat...,True,19.54
2,MuterBandung_Behavior_Model_Workspace\03_Curat...,True,2.90
3,MuterBandung_Behavior_Model_Workspace\04_Evalu...,True,0.02
4,MuterBandung_Behavior_Model_Workspace\04_Evalu...,True,0.01
5,MuterBandung_Behavior_Model_Workspace\04_Evalu...,True,0.00


Ringkasan: tahap filtering dan mapping selesai. Langkah berikutnya adalah review kategori needs_review, lalu training baseline next-category.